In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [2]:
jobs = pd.read_csv("../data/processed/cleaned_jobs.csv")
print("Job dataset loaded successfully!")
print("Number of jobs:", len(jobs))
print("Number of columns:", len(jobs.columns))
jobs.head()

Job dataset loaded successfully!
Number of jobs: 61773
Number of columns: 13


,job_id,job_role,company,experience,salary,location,rating,reviews,key_skills,posted_on,job_link,company_link,job_text
0,181224917811,GN - Strategy - MC - SC&O - SC Digital Core Ar...,Accenture,3-6 Yrs,Not disclosed,"Hyderabad, Chennai, Bengaluru",3.9,52479 Reviews,"sap ariba,presentation skills,solution impleme...",1 Day Ago,https://www.naukri.com/job-listings-gn-strateg...,https://www.naukri.com/accenture-jobs-careers-...,GN - Strategy - MC - SC&O - SC Digital Core Ar...
1,181224906322,GN - SC&O - BPM - Senior Manager,Accenture,2-4 Yrs,Not disclosed,"Gurugram, Bengaluru, Delhi / NCR",3.9,52479 Reviews,"spend analysis,market research,sales and opera...",1 Day Ago,https://www.naukri.com/job-listings-gn-sc-o-bp...,https://www.naukri.com/accenture-jobs-careers-...,GN - SC&O - BPM - Senior Manager spend analysi...
2,181224906320,GN - SC&O - S&P - CLM - Associate Manager,Accenture,1-5 Yrs,Not disclosed,"Gurugram, Bengaluru, Delhi / NCR",3.9,52479 Reviews,"Sourcing,PowerBI,procurement,digital sourcing,...",1 Day Ago,https://www.naukri.com/job-listings-gn-sc-o-s-...,https://www.naukri.com/accenture-jobs-careers-...,GN - SC&O - S&P - CLM - Associate Manager Sour...
3,181224904080,GN - SONG - Service - Command Center of Future...,Accenture,12-15 Yrs,Not disclosed,"New Delhi, Gurugram, Bengaluru",3.9,52479 Reviews,"cloud solutions,WFM solutions,AI,Customer Enga...",1 Day Ago,https://www.naukri.com/job-listings-gn-song-se...,https://www.naukri.com/accenture-jobs-careers-...,GN - SONG - Service - Command Center of Future...
4,181224904079,GN - SC&O - S&P - CLM - Manager,Accenture,8-13 Yrs,Not disclosed,"New Delhi, Gurugram, Bengaluru",3.9,52479 Reviews,"Procurement,Sourcing,Supply Chain Management,p...",1 Day Ago,https://www.naukri.com/job-listings-gn-sc-o-s-...,https://www.naukri.com/accenture-jobs-careers-...,"GN - SC&O - S&P - CLM - Manager Procurement,So..."


In [3]:
job_text = jobs["job_text"].fillna("")
print("Job text prepared successfully!")
print("Number of job descriptions:", len(job_text))
print("\nSample job text:")
print(job_text.iloc[0][:500])

Job text prepared successfully!
Number of job descriptions: 61773

Sample job text:
GN - Strategy - MC - SC&O - SC Digital Core Ariba - Consultant sap ariba,presentation skills,solution implementation,ariba,tower,management consulting,project management,sap 3-6 Yrs Hyderabad, Chennai, Bengaluru


In [4]:
topic_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)
topic_matrix = topic_tfidf.fit_transform(job_text)
print("Topic TF-IDF created successfully!")
print("Topic matrix shape:", topic_matrix.shape)

Topic TF-IDF created successfully!
Topic matrix shape: (61773, 5000)


In [5]:
nmf_model = NMF(
    n_components=8,
    random_state=42,
    init="nndsvda",
    max_iter=200
)
topic_matrix_result = nmf_model.fit_transform(topic_matrix)
print("NMF Topic Modeling completed successfully!")
print("Number of topics:", 8)
print("Topic matrix shape:", topic_matrix_result.shape)

NMF Topic Modeling completed successfully!
Number of topics: 8
Topic matrix shape: (61773, 8)


In [6]:
feature_names = topic_tfidf.get_feature_names_out()
print("Top words in each topic:\n")
for topic_index, topic in enumerate(nmf_model.components_):
    top_words = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_words]
    print("Topic", topic_index + 1, ":", ", ".join(words))

Top words in each topic:

Topic 1 : sales, executive, field, business, channel, development, manager, b2b, area, relationship
Topic 2 : management, project, manager, operations, business, yrs, team, skills, risk, planning
Topic 3 : process, customer, voice, international, service, bpo, support, inbound, chat, non
Topic 4 : sap, abap, hana, application, fico, consultant, sd, mm, implementation, basis
Topic 5 : development, developer, application, software, java, css, web, net, javascript, spring
Topic 6 : data, analysis, learning, machine, python, analytics, sql, analyst, algorithms, modeling
Topic 7 : marketing, digital, media, executive, social, content, seo, campaigns, google, brand
Topic 8 : testing, automation, engineer, software, quality, test, design, engineering, bengaluru, selenium


In [7]:
import os
import joblib
os.makedirs("../models", exist_ok=True)
joblib.dump(topic_tfidf, "../models/topic_tfidf.pkl")
joblib.dump(nmf_model, "../models/nmf_topic_model.pkl")
print("Topic TF-IDF model saved successfully!")
print("NMF topic model saved successfully!")

Topic TF-IDF model saved successfully!
NMF topic model saved successfully!


In [8]:
topic_names = {
    1: "Sales & Business Development",
    2: "Management & Operations",
    3: "BPO & Customer Support",
    4: "SAP & Enterprise Consulting",
    5: "Software Development",
    6: "Data Science & Analytics",
    7: "Digital Marketing & Media",
    8: "Software Testing & Quality Engineering"
}
topic_summary = []
for topic_number, topic in enumerate(nmf_model.components_, start=1):
    top_words = topic.argsort()[-10:][::-1]
    words = [
        feature_names[i]
        for i in top_words
    ]
    topic_summary.append({
        "Topic": topic_number,
        "Topic Name": topic_names[topic_number],
        "Top Words": ", ".join(words)
    })
topic_summary_df = pd.DataFrame(topic_summary)
topic_summary_df

,Topic,Topic Name,Top Words
0,1,Sales & Business Development,"sales, executive, field, business, channel, de..."
1,2,Management & Operations,"management, project, manager, operations, busi..."
2,3,BPO & Customer Support,"process, customer, voice, international, servi..."
3,4,SAP & Enterprise Consulting,"sap, abap, hana, application, fico, consultant..."
4,5,Software Development,"development, developer, application, software,..."
5,6,Data Science & Analytics,"data, analysis, learning, machine, python, ana..."
6,7,Digital Marketing & Media,"marketing, digital, media, executive, social, ..."
7,8,Software Testing & Quality Engineering,"testing, automation, engineer, software, quali..."
